# Étape 4b — Balayage du rang

**Projet 5 · LoRA Zellige** — Google Colab, GPU T4.

Quatre entraînements, rangs 4, 8, 16 et 32, **à budget strictement
identique** : 400 pas, accumulation 2. Seul le rang change.

Ce budget est plus court que celui de l'Étape 3 (1000 pas, accumulation 4,
49 minutes). Les quatre rangs restent comparables **entre eux**, mais pas
avec `lora_r8_1000pas`. C'est un choix assumé, imposé par le temps GPU
disponible, et il doit être écrit tel quel dans le rapport.

Durée : environ 1 heure. Lancez-le quand vous pouvez laisser la session
ouverte.

> **Exécution → Modifier le type d'exécution → GPU T4**


## 1. Environnement


In [ ]:
!pip -q uninstall -y torchao 2>&1 | tail -1
!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors bitsandbytes lpips pyyaml 2>&1 | tail -2
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())


## 2. Dépôt et jeu de données


In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet 2>/dev/null || (cd /content/projet && git fetch -q && git reset -q --hard origin/main)
%cd /content/projet
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py


In [ ]:
import subprocess

def lancer(commande):
    """Execute une commande et s'arrete net si elle echoue."""
    print('>>', commande, flush=True)
    code = subprocess.run(commande, shell=True).returncode
    assert code == 0, f'echec (code {code}) : {commande}'


## 3. Test rapide

Deux minutes pour vérifier que la chaîne tourne avant d'engager une heure.


In [ ]:
lancer('python src/train_lora.py --config configs/lora_r4_400.yaml --pas 10 --sortie runs/test')
from pathlib import Path
assert list(Path('runs/test').glob('*.safetensors')), 'chaine cassee'
print('chaine validee')


## 4. Les quatre entraînements

Ce que le rang change concrètement : c'est la dimension interne des deux
petites matrices `A` et `B` qu'on apprend à la place de `W`. Rang 4 =
~800 000 paramètres ; rang 32 = ~6,4 millions.

L'hypothèse : **plus de rang = plus de style, mais plus de mémorisation et
moins d'obéissance au texte.** C'est ce que la figure doit montrer — ou
démentir.

~10 min d'entraînement + ~4 min de mesure par rang, soit ~1 heure.


In [ ]:
for rang in [4, 8, 16, 32]:
    nom = f'lora_r{rang}_400pas'
    lancer(f'python src/train_lora.py --config configs/lora_r{rang}_400.yaml')
    lancer(f'python src/generate.py --lora runs/r{rang}_400 --intensite 1.0 --sortie results/r{rang}_400')
    lancer(f'python src/evaluate.py --dossier results/r{rang}_400 --nom {nom} --rang {rang} --pas 400')
    print(f'=== rang {rang} termine ===', flush=True)


## 5. Le coût de chaque rang

La réponse chiffrée à la question Q3 du projet.


In [ ]:
import json, pandas as pd
lignes = []
for rang in [4, 8, 16, 32]:
    r = json.load(open(f'runs/r{rang}_400/entrainement.json'))
    lignes.append({'rang': rang,
                   'parametres': r['parametres_entrainables'],
                   'part_%': r['part_entrainable_pct'],
                   'duree_min': round(r['duree_s'] / 60, 1),
                   'VRAM_Go': round(r['vram_pic_go'], 2),
                   'fichier_Mo': r['taille_lora_mo'],
                   'perte_finale': r['perte_finale']})
cout = pd.DataFrame(lignes).set_index('rang')
cout.to_csv('results/cout_par_rang.csv')
cout


## 6. Les figures


In [ ]:
!python src/figures.py

from IPython.display import Image, display
for f in ['rangs.png', 'compromis.png']:
    print('—' * 70); print(f)
    display(Image(f'results/figures/{f}'))


## 7. La planche des rangs

Même prompt, même graine, quatre rangs. Le surapprentissage se voit souvent
à l'œil avant d'apparaître dans les chiffres.


In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

rangs = [4, 8, 16, 32]
lignes = ['D1', 'D3', 'T1']
fig, axes = plt.subplots(len(lignes), len(rangs),
                         figsize=(2.8 * len(rangs), 2.9 * len(lignes)))
for i, pid in enumerate(lignes):
    for j, rang in enumerate(rangs):
        ax = axes[i, j]
        ax.imshow(PILImage.open(f'results/r{rang}_400/{pid}_s1234.png'))
        ax.axis('off')
        if i == 0: ax.set_title(f'rang {rang}', fontsize=11)
        if j == 0: ax.text(-0.08, 0.5, pid, transform=ax.transAxes,
                           rotation=90, va='center', fontsize=11, weight='bold')
plt.suptitle('Effet du rang a budget identique - graine 1234', fontsize=13)
plt.tight_layout(); plt.savefig('results/figures/planche_rangs.png', dpi=110)
plt.show()


## 8. Le tableau complet


In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
df = pd.read_csv('results/results.csv')
df[['nom', 'rang', 'pas', 'intensite', 'style_dino_moy', 'prompt_domaine',
    'prompt_transfert', 'prompt_hors_domaine', 'memorisation_max_dino',
    'diversite_lpips']]


## 9. Sauvegarder


In [ ]:
from google.colab import userdata
jeton = userdata.get('GITHUB_TOKEN')
url = f'https://{jeton}@github.com/HatimTj/lora-zellige-patrimoine.git'

!git config user.name 'HatimTj'
!git config user.email 'hatimtajimi1@gmail.com'
!git add results/results.csv results/cout_par_rang.csv results/figures/*.png
!git add results/*/manifeste.json results/*/evaluation.json runs/*/entrainement.json
!git commit -m 'etape4: balayage du rang a budget identique'
!git pull --rebase -q {url} main
!git push {url} main


## 10. Récupérer les LoRA

Quatre fichiers, de 3 à 24 Mo. Celui du meilleur rang servira à la démo.


In [ ]:
!zip -qr lora_rangs.zip runs/r4_400 runs/r8_400 runs/r16_400 runs/r32_400
from google.colab import files
files.download('lora_rangs.zip')
